In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import MaxNLocator
from parametric_impact_pinn_tf1 import train_parametric_pinn
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # CPU:-1; GPU0: 0

In [ ]:
np.random.seed(1234)
tf.compat.v1.set_random_seed(1234)
print('TensorFlow version: {}'.format(tf.__version__))

## 2. Train Parametric PINN across 50 parameter cases (Latin Hypercube Sampling)

In [ ]:
layers   = [10, 64, 64, 64, 1]
n_impact = 4

# Normalization bounds for the 9 non-time input channels:
# [mx,  my,   k,    c,    D,   y0,   yt0,   x0,   xt0]
lb_params = [0.5,  0.1,  0.5,  0.0,  0.5, -0.1, -1.5, -0.1, -0.1]
ub_params = [2.0,  0.6,  2.0,  0.1,  2.0,  0.1, -0.5,  0.1,  0.1]

# Fix initial conditions so the network trains over the physical-parameter space.
# Remove fixed_ic (or extend its keys) to also vary ICs parametrically.
fixed_ic = {'x0': 0.0, 'xt0': 0.0, 'y0': 0.0, 'yt0': -1.0}

predictor, seg_models = train_parametric_pinn(
    n_cases=50,
    n_impacts=n_impact,
    lb_params=lb_params,
    ub_params=ub_params,
    r=1.0,
    fixed_ic=fixed_ic,
    layers=layers,
    T_max_per_segment=[3.0, 5.0, 5.0, 8.0],
    nIter_per_segment=[1000, 1000, 1000, 1000],
    n_r=200,
    hyp_ini_para=0.5,
    hyp_ini_weight_loss=(1.0, 1.0, 10.0),
    optimizer_LB=True,
    seed=42,
)

In [ ]:
# Reference case: predict for a specific parameter set (not one of the 50 training samples).
# This uses only a forward pass through the trained network + root-finding — no retraining.
ref_mx, ref_my, ref_k, ref_c, ref_D = 1.0, 0.3, 1.0, 0.0, 1.0

results = predictor.predict(
    mx=ref_mx, my=ref_my, k=ref_k, c=ref_c, D=ref_D,
    x0=0.0, xt0=0.0, y0=0.0, yt0=-1.0,
    num_points=1000,
)

## 3. Extract results

In [ ]:
all_time = results['time']
all_x    = results['x']
all_xt   = results['xt']
all_y    = results['y']
all_yt   = results['yt']

impact_times       = results['impact_times']        # per-segment durations
impact_time_stamps = np.cumsum(impact_times)        # cumulative, for plot markers

print('Learned impact times (per segment):', np.round(impact_times, 4))
print('Cumulative (PINN):                 ', np.round(impact_time_stamps, 4))

## 4. Plot — Deflection vs Time

In [ ]:
mpl.rcParams['font.family'] = 'Times New Roman'
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Times New Roman'
mpl.rcParams['mathtext.it'] = 'Times New Roman:italic'
mpl.rcParams['mathtext.bf'] = 'Times New Roman:bold'
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

fontsize_labels = 28
fontsize_ticks  = 28
fontsize_legend = 24
linewidth_plot  = 2
linewidth_vline = 1

plt.figure(figsize=(10, 4.5))
plt.plot(all_time, all_x, label=r'$x(t)$', linewidth=linewidth_plot)
plt.plot(all_time, all_y, label=r'$y(t)$', linestyle='--', linewidth=linewidth_plot)
for t_imp in impact_time_stamps:
    plt.axvline(x=t_imp, color='gray', linestyle='--', linewidth=linewidth_vline)
plt.xlabel('Time (t)', fontsize=fontsize_labels, labelpad=8)
plt.ylabel('Deflection (m)', fontsize=fontsize_labels, labelpad=12)
ax = plt.gca()
ax.xaxis.set_major_locator(MaxNLocator(nbins=10, prune='both'))
ax.yaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
ax.tick_params(axis='both', labelsize=fontsize_ticks)
plt.legend(fontsize=fontsize_legend, loc='upper right')
plt.tight_layout(pad=1.5)
plt.xlim(0, impact_time_stamps[-1] * 1.05)
plt.ylim(-1.8, 1.8)
plt.savefig('deflection_vs_time.png', format='png', dpi=300)
plt.show()

## 5. Plot — Velocity vs Time

In [ ]:
plt.figure(figsize=(10, 4.5))
plt.plot(all_time, all_xt, label=r'$\dot{x}(t)$', linewidth=linewidth_plot)
plt.plot(all_time, all_yt, label=r'$\dot{y}(t)$', linestyle='--', linewidth=linewidth_plot)
for t_imp in impact_time_stamps:
    plt.axvline(x=t_imp, color='gray', linestyle='--', linewidth=linewidth_vline)
plt.xlabel('Time (t)', fontsize=fontsize_labels, labelpad=8)
plt.ylabel('Velocity (m/s)', fontsize=fontsize_labels, labelpad=12)
ax = plt.gca()
ax.xaxis.set_major_locator(MaxNLocator(nbins=10, prune='both'))
ax.yaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
ax.tick_params(axis='both', labelsize=fontsize_ticks)
plt.legend(fontsize=fontsize_legend, loc='upper right')
plt.tight_layout(pad=1.5)
plt.xlim(0, impact_time_stamps[-1] * 1.05)
plt.ylim(-1.4, 1.4)
plt.savefig('velocity_vs_time.png', format='png', dpi=300)
plt.show()

## 6. Plot — Relative Deflection (x - y)

In [ ]:
plt.figure(figsize=(10, 4.5))
plt.plot(all_time, all_x - all_y, label='PINN', linestyle='--', linewidth=linewidth_plot)
plt.axhline(y= 1.0, color='black', linestyle='-.', linewidth=2)
plt.axhline(y=-1.0, color='black', linestyle='-.', linewidth=2)
for t_imp in impact_time_stamps:
    plt.axvline(x=t_imp, color='gray', linestyle='--', linewidth=linewidth_vline)
plt.xlabel('Time (t)', fontsize=fontsize_labels, labelpad=8)
plt.ylabel('Relative Deflection (m)', fontsize=fontsize_labels, labelpad=10)
ax = plt.gca()
ax.xaxis.set_major_locator(MaxNLocator(nbins=10, prune='both'))
ax.yaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
ax.tick_params(axis='both', labelsize=fontsize_ticks)
plt.legend(fontsize=fontsize_legend, loc='lower right')
plt.tight_layout(pad=1.5)
plt.xlim(0, impact_time_stamps[-1] * 1.05)
plt.ylim(-1.4, 1.4)
plt.savefig('relative_deflection.png', format='png', dpi=300)
plt.show()

## 7. Plot — Impulse

In [ ]:
delta_yt = np.diff(all_yt, axis=0)
time_for_delta_yt = all_time[:-1]
impulse = ref_my * delta_yt

plt.figure(figsize=(10, 4.5))
plt.plot(time_for_delta_yt, impulse,
         label=r'$m_y(\dot{y}_{n+1} - \dot{y}_n)$', linewidth=linewidth_plot)
for t_imp in impact_time_stamps:
    plt.axvline(x=t_imp, color='gray', linestyle='--', linewidth=linewidth_vline)
plt.xlabel('Time (t)', fontsize=fontsize_labels, labelpad=8)
plt.ylabel('Impulse (Ns)', fontsize=fontsize_labels, labelpad=10)
ax = plt.gca()
ax.xaxis.set_major_locator(MaxNLocator(nbins=10, prune='both'))
ax.yaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
ax.tick_params(axis='both', labelsize=fontsize_ticks)
plt.legend(fontsize=fontsize_legend)
plt.tight_layout(pad=1.5)
plt.xlim(0, impact_time_stamps[-1] * 1.05)
plt.ylim(-0.6, 0.6)
plt.savefig('impulse_plot.png', format='png', dpi=300)
plt.show()

## 8. Plot — Impact Time Convergence per Segment

In [ ]:
for i, model in enumerate(seg_models):
    # lambda_1_log[k] has shape (n_cases, 1); plot mean across training cases
    lambda_log_mean = [float(np.mean(v)) for v in model.lambda_1_log]
    lambda_log_std  = [float(np.std(v))  for v in model.lambda_1_log]
    iters = range(len(lambda_log_mean))

    plt.figure(figsize=(4.5, 4.5))
    plt.plot(iters, lambda_log_mean, linewidth=2, label='Mean over cases')
    plt.fill_between(iters,
                     np.array(lambda_log_mean) - np.array(lambda_log_std),
                     np.array(lambda_log_mean) + np.array(lambda_log_std),
                     alpha=0.25, label='±1 std')
    plt.xlabel('Iteration', fontsize=24, labelpad=10)
    plt.ylabel('Impact time', fontsize=24, labelpad=10)
    plt.title(f'Segment {i+1}', fontsize=20)
    ax = plt.gca()
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4, prune='both'))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4, prune='both'))
    ax.tick_params(axis='both', labelsize=24)
    plt.legend(fontsize=14)
    plt.tight_layout(pad=1.5)
    plt.savefig(f'impact_convergence_{i+1}.png', format='png', dpi=300)
    plt.show()

## 9. Plot — Training Loss (last segment)

In [ ]:
# To inspect a different segment, change the index: seg_models[i]
model = seg_models[-1]

fig, ax = plt.subplots(figsize=(5, 2.7))
ax.loglog(model.loss_log,          label='Total loss')
ax.loglog(model.loss_icx_log,      label='Loss_IC')
ax.loglog(model.loss_fx_log,       label='Loss_ODE')
ax.loglog(model.loss_f_log,        label='Loss_impact')
ax.loglog(model.loss_lambda_box_log, label='Loss_λ box')
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Parametric Predictions — Three New Parameter Sets (No Retraining)

In [ ]:
# The predictor trained in Section 2 generalises over the full parameter space.
# Calling predictor.predict() runs only a forward pass + root-finding — no optimisation.
test_cases = [
    dict(mx=1.0, my=0.3,  k=1.0,  c=0.0,  D=1.0,  label='Case A  (mx=1.0, my=0.3,  k=1.0,  D=1.0)'),
    dict(mx=1.5, my=0.5,  k=1.8,  c=0.05, D=0.7,  label='Case B  (mx=1.5, my=0.5,  k=1.8,  D=0.7)'),
    dict(mx=0.7, my=0.15, k=0.6,  c=0.0,  D=1.5,  label='Case C  (mx=0.7, my=0.15, k=0.6,  D=1.5)'),
]

results_test = []
for tc in test_cases:
    res = predictor.predict(
        mx=tc['mx'], my=tc['my'], k=tc['k'], c=tc['c'], D=tc['D'],
        x0=0.0, xt0=0.0, y0=0.0, yt0=-1.0,
        num_points=500,
    )
    results_test.append(res)
    print(f"{tc['label']}")
    print(f"  Impact times: {np.round(res['impact_times'], 4)}")
    print()

## 11. Predict for New Parameter Sets — No Retraining

In [ ]:
# Three NEW parameter sets — none of these were training cases.
# The predictor uses the trained network + root-finding only.
test_cases = [
    dict(mx=1.0, my=0.3,  k=1.0,  c=0.0,  D=1.0,  label='Case A  (mx=1.0, my=0.3, k=1.0, D=1.0)'),
    dict(mx=1.5, my=0.5,  k=1.8,  c=0.05, D=0.7,  label='Case B  (mx=1.5, my=0.5, k=1.8, D=0.7)'),
    dict(mx=0.7, my=0.15, k=0.6,  c=0.0,  D=1.5,  label='Case C  (mx=0.7, my=0.15, k=0.6, D=1.5)'),
]

results_test = []
for tc in test_cases:
    res = predictor.predict(
        mx=tc['mx'], my=tc['my'], k=tc['k'], c=tc['c'], D=tc['D'],
        x0=0.0, xt0=0.0, y0=0.0, yt0=-1.0,
        num_points=500,
    )
    results_test.append(res)
    print(f"{tc['label']}")
    print(f"  Impact times: {np.round(res['impact_times'], 4)}")
    print()

## 12. Plot — Parametric Predictions for Three Different Parameter Sets

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=False)

for ax, res, tc in zip(axes, results_test, test_cases):
    t_end = res['impact_times'].sum()
    t_stamps = np.cumsum(res['impact_times'])

    ax.plot(res['time'], res['x'], label=r'$x(t)$ — main mass', linewidth=2)
    ax.plot(res['time'], res['y'], '--', label=r'$y(t)$ — impactor', linewidth=2)
    for ts in t_stamps:
        ax.axvline(x=ts, color='gray', linestyle=':', linewidth=1)
    ax.set_title(tc['label'], fontsize=14)
    ax.set_xlabel('Time', fontsize=13)
    ax.set_ylabel('Displacement', fontsize=13)
    ax.legend(fontsize=12)
    ax.set_xlim(0, t_end * 1.05)
    ax.tick_params(labelsize=12)

plt.suptitle('True Parametric PINN — Instant Prediction for New Parameters\n'
             '(no retraining, only forward pass + root-finding)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('parametric_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Stage 2 — Dispersion Curve of the Meta-impactor Lattice

**What is a dispersion curve?**

A dispersion curve plots the relationship between:
- $k$ — **wavenumber** (number of wave cycles per lattice spacing, rad/unit cell)
- $\omega$ — **angular frequency** (rad/s)

It answers: *"For waves of a given spatial wavelength, how fast do they oscillate?"*

**Algorithm — 2-D FFT method:**

1. Simulate a ring of $N$ coupled unit cells for time $T$:
$$M\ddot{x}_n + K(2x_n - x_{n-1} - x_{n+1}) = F_{\text{impact},n}, \qquad n=0,\ldots,N-1$$

2. Apply 2-D FFT over the spatiotemporal field $x_n(t)$:
$$S(k,\omega) = \left|\text{FFT}_{2D}\bigl[x_n(t)\bigr]\right|^2$$

3. Plot $S(k,\omega)$ as a heatmap. The dispersion relation $\omega(k)$ appears as **bright ridges**.

The linear (no-impactor) acoustic branch: $\omega_{\text{lin}}(k) = 2\sqrt{K/M}\,|\sin(k/2)|$

## 13. Compute Dispersion Curve — Wavenumber Sweep

In [ ]:
from dispersion_curve import simulate_lattice, dispersion_from_2dfft

# -------------------------------------------------------------------
# Lattice parameters
# -------------------------------------------------------------------
N          = 32        # number of unit cells in the ring
mx_lat     = 1.0       # primary mass M
my_lat     = 0.3       # impactor mass m
k_coupling = 1.0       # inter-cell spring stiffness K
c_lat      = 0.0       # damping
D_lat      = 1.0       # impact gap
r_lat      = 1.0       # restitution coefficient
T_sim      = 120.0     # simulation time (needs several wave periods)
n_steps    = 12000     # time steps  →  dt = T_sim / n_steps

# Broadband IC ('random') excites ALL wavenumbers simultaneously
# so ONE simulation gives the full dispersion curve.
# Use yt0_magnitude to control the excitation amplitude.

print('Simulating lattice …')
t_lat, x_nt = simulate_lattice(
    N          = N,
    mx         = mx_lat,
    my         = my_lat,
    k_coupling = k_coupling,
    c          = c_lat,
    D          = D_lat,
    r          = r_lat,
    T_sim      = T_sim,
    n_steps    = n_steps,
    ic_type    = 'random',        # broadband: all k excited
    yt0_magnitude = 1.0,
    seed       = 42,
)
print(f'Done.  x_nt shape: {x_nt.shape}  (N cells × T time steps)')

# 2-D FFT  →  frequency–wavenumber spectrum
k_vals, omega_vals, spectrum = dispersion_from_2dfft(t_lat, x_nt, skip_transient=0.2)
print(f'k range:  [{k_vals.min():.3f}, {k_vals.max():.3f}] rad/cell')
print(f'ω range:  [{omega_vals.min():.3f}, {omega_vals.max():.3f}] rad/s')

## 14. Plot — Dispersion Curve

In [ ]:
from dispersion_curve import (plot_dispersion_heatmap, extract_ridge,
                               plot_dispersion_amplitude, simulate_lattice,
                               dispersion_from_2dfft)

# ── Heatmap for amplitude = 1.0 ──────────────────────────────────────────────
plot_dispersion_heatmap(
    k_vals, omega_vals, spectrum,
    k_coupling = k_coupling,
    mx         = mx_lat,
    dB_range   = 40.0,
    save_path  = 'dispersion_heatmap.png',
)

# ── Amplitude sweep: ridge extraction for three excitation levels ─────────────
amp_sweep = [0.3, 1.0, 2.0]
ridge_results = []

for amp in amp_sweep:
    print(f'Simulating for amplitude = {amp} …')
    t_a, x_a = simulate_lattice(
        N=N, mx=mx_lat, my=my_lat, k_coupling=k_coupling,
        c=c_lat, D=D_lat, r=r_lat,
        T_sim=T_sim, n_steps=n_steps,
        ic_type='random', yt0_magnitude=amp, seed=42,
    )
    k_a, om_a, spec_a = dispersion_from_2dfft(t_a, x_a, skip_transient=0.2)
    k_ridge, om_ridge = extract_ridge(k_a, om_a, spec_a)
    ridge_results.append({'k': k_ridge, 'omega': om_ridge,
                           'label': rf'$|\dot{{y}}_0|={amp}$'})

plot_dispersion_amplitude(
    ridge_results,
    k_coupling = k_coupling,
    mx         = mx_lat,
    save_path  = 'dispersion_amplitude_sweep.png',
)